In [84]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_ollama.llms import OllamaLLM

from langchain_core.tools import tool

llm=ChatGroq(model="qwen-2.5-32b")
# llm = OllamaLLM(model="llama3.2:1B")

In [85]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


## Multi-Step Reasoning in Agentic AI (Dynamic Step Determination)

Unlike static Chain-of-Thought (CoT), where reasoning steps are predefined, Agentic AI dynamically decides the necessary steps based on the problem at hand.

💡 Example Use Case:
An AI assistant needs to book a flight, find hotel options, and generate a trip summary dynamically. Depending on the user's query, it may take different steps.

In [86]:
# we can replace these with api calls
from langchain.agents import initialize_agent, Tool
from langchain.chat_models import ChatOpenAI
from langchain.utilities import SerpAPIWrapper
import requests

# LLM Model (can be GPT-4, Llama 3, etc.)
# llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

# Tool 1: Flight Search API
@tool
def search_flights(destination: str, date: str):
    """use this tool to Fetch available flights.
    Args:
        destination : name of the place
        date : travel date in dd/mm/yyyy format.

    Returns:
        str: returns flight info
    """
    return f"Flights found to {destination} on {date}: Airline X - $400, Airline Y - $350."

# Tool 2: Hotel Search API
@tool
def search_hotels(destination: str):
    """use this tool to Find available hotels.
    
    Args:
        destination : name of the place
    
    Returns:
        str: lists all the hotels 
        
    """
    return f"Available hotels in {destination}: Hotel A - $120/night, Hotel B - $150/night."

# Tool 3: Generate Trip Summary
@tool
def generate_trip_summary(destination: str, flight_info: str, hotel_info: str):
    """use this tool to Generate a trip itinerary.
    
    Args:
        destination : name of the place
        flight_info : flights details
        hotel_info : hotel information

    Returns:
        str: Trip summary details.
    
    """
    return f"Trip Summary:\n- Destination: {destination}\n- {flight_info}\n- {hotel_info}\n- Recommended activities: Sightseeing, local food tour."

# # Define Tools
# flight_tool = Tool(name="Flight Search", func=search_flights, description="Finds flight options.")
# hotel_tool = Tool(name="Hotel Search", func=search_hotels, description="Finds hotel options.")
# summary_tool = Tool(name="Trip Summary", func=generate_trip_summary, description="Creates a travel plan.")

# tools = [flight_tool, hotel_tool, summary_tool]
tools = [search_flights, search_hotels, generate_trip_summary]


- ZERO_SHOT_REACT_DESCRIPTION = 'zero-shot-react-description'

    A zero shot agent that does a reasoning step before acting.


In [87]:
# # Initialize Agent with Dynamic Step Selection
# agent = initialize_agent(
#     tools=tools,
#     llm=llm,
#     agent="zero-shot-react-description",  # Allows dynamic step selection
#     verbose=True,
#     handle_parsing_errors=True, 
#     # model = llm.bind_tools(tools)
# )

# # User Query
# query = "Plan a trip to Tokyo for 25-05-2025"
# response = agent.run(query)
# print("Final Response:", response)


In [88]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# Define the graph

from langgraph.prebuilt import create_react_agent

graph = create_react_agent(llm, tools=tools, checkpointer=memory)

In [89]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [90]:
config = {"configurable": {"thread_id": "1"}}
inputs = {"messages": [("user", "Plan a trip to Tokyo for 25-05-2025")]}

print_stream(graph.stream(inputs, config=config, stream_mode="values"))


================================ Human Message =================================

Plan a trip to Tokyo for 25-05-2025
================================== Ai Message ==================================
Tool Calls:
  search_flights (call_qqdb)
 Call ID: call_qqdb
  Args:
    destination: Tokyo
    date: 25/05/2025
  search_hotels (call_2zz6)
 Call ID: call_2zz6
  Args:
    destination: Tokyo
================================= Tool Message =================================
Name: search_hotels

Available hotels in Tokyo: Hotel A - $120/night, Hotel B - $150/night.
================================== Ai Message ==================================

Based on the information provided, here are your options for a trip to Tokyo on 25/05/2025:

- **Flight Options:**
  - Airline X for $400
  - Airline Y for $350

- **Hotel Options:**
  - Hotel A for $120 per night
  - Hotel B for $150 per night

Would you like to proceed with one of these options to generate a trip summary, or do you need more informat